In [ ]:
import sys
from pathlib import Path

# Repo root (parent of notebooks/) so `utils` and `src` import the same way the scripts do
REPO_ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(REPO_ROOT))

import base64
import warnings

import numpy as np
import polars as pl
import rasterio
from ipyleaflet import (
    FullScreenControl,
    ImageOverlay,
    LayersControl,
    Map,|
    Marker,
    basemaps,
)
from ipywidgets import Layout
from rasterio.enums import Resampling
from rasterio.errors import NotGeoreferencedWarning
from rasterio.io import MemoryFile
from rasterio.vrt import WarpedVRT
from rasterio.warp import transform_bounds

from utils.variables import (
    ALOS2_ATTRIBUTION,
    ALOS2_GEOTIFF_PATH,
    ALOS2_GRANULE_CSV_PATH,
    GRANULE_CSV_PATH,
    NISAR_GEOTIFF_PATH,
    SEARCH_AOIS,
)

In [ ]:
AOI = "leaky-infrastructure-70111"  # AOI from SEARCH_AOIS (utils/aois.py); its point is shown as a marker
# Folder the GeoTIFFs were written to (--out-dir), e.g. "data/timing_nisar_w4" or "data/timing_alos2_w4".
# None = the defaults, data/nisar_geotiffs and data/alos2_geotiffs
OUT_DIR = None
MAX_PX = 2048  # longest side of each rendered image; keeps the notebook light

# Where to look for each sensor's GeoTIFFs; relative paths are from the repo root, as in the *_access commands
if OUT_DIR is None:
    tif_dirs = {"NISAR": NISAR_GEOTIFF_PATH, "ALOS-2": ALOS2_GEOTIFF_PATH}
else:
    tif_dirs = dict.fromkeys(("NISAR", "ALOS-2"), REPO_ROOT / OUT_DIR)
    if not tif_dirs["NISAR"].is_dir():
        raise FileNotFoundError(f"{tif_dirs['NISAR']} does not exist")
csv_dirs = {"NISAR": GRANULE_CSV_PATH, "ALOS-2": ALOS2_GRANULE_CSV_PATH}


def acquisition_date(tif):
    """YYYYMMDD from a NISAR or ALOS-2 GeoTIFF name."""
    if tif.name.startswith("ALOS2"):
        # e.g. ALOS2660973500-260819_WBDR2.2GUD_HH_SLP_gamma0_... -> 260819
        return "20" + tif.name.split("-")[1][:6]
    # e.g. NISAR_L2_PR_GCOV_..._A_20261020T175614_... -> 20261020
    return tif.name.split("_")[11][:8]


def scene_label(sensor, tif):
    """Short name for the layers control, e.g. 'NISAR 2026-10-20 123D/086 DHDH freqA HHHH' or 'ALOS-2 2026-08-19 frame 3500 HH'."""
    d = acquisition_date(tif)
    date = f"{d[:4]}-{d[4:6]}-{d[6:]}"
    parts = tif.stem.split("_")
    if sensor == "NISAR":
        # track + direction / frame, polarization mode (DHDH dual, SHSH single), then frequency + polarization
        return f"NISAR {date} {parts[5]}{parts[6]}/{parts[7]} {parts[9]} {parts[18]} {parts[19]}"
    # ALOS-2 scene ID is ALOS2 + 5-digit orbit + 4-digit frame; then polarization, and whether it's masked
    return f"ALOS-2 {date} frame {parts[0][10:14]} {parts[2]}" + (" masked" if "_masked" in tif.stem else "")


def cropped_to(tif):
    """AOI a GeoTIFF was cropped to (from its _<aoi name> suffix), or None for a full scene."""
    return next((a for a in SEARCH_AOIS if tif.stem.endswith(f"_{a}")), None)


def latest_request_tifs(sensor):
    """GeoTIFFs in this sensor's folder for the granules in its most recent search for AOI."""
    csvs = list(csv_dirs[sensor].glob(f"*_{AOI}_granules_*.csv"))
    if not csvs:
        print(f"{sensor}: no searches for {AOI}")
        return []
    # Timestamp after "requestMade" sorts chronologically
    request_csv = max(csvs, key=lambda p: p.stem.split("_requestMade")[-1])
    granules = pl.read_csv(request_csv)["Granule Name"].unique().to_list()
    # GeoTIFFs are named after their granule, so match on that prefix; skip crops made for other AOIs (full scenes are kept)
    tifs = [
        t
        for g in granules
        for t in tif_dirs[sensor].glob(f"{g}_*.tif")
        if cropped_to(t) in (None, AOI)
    ]
    print(f"{sensor}: {request_csv.name}\n  {len(granules)} granules in request, {len(tifs)} GeoTIFFs in {tif_dirs[sensor]}")
    return [(sensor, t) for t in tifs]


# Oldest -> newest acquisition across both sensors; on the same date, crops come after full scenes so they're drawn on top
scenes = sorted(
    latest_request_tifs("NISAR") + latest_request_tifs("ALOS-2"),
    key=lambda s: (acquisition_date(s[1]), cropped_to(s[1]) is not None, s[1].name),
)
if not scenes:
    raise FileNotFoundError("No GeoTIFFs for the latest searches in that folder yet; run nisar_access / alos2_access first")


def tif_to_overlay(sensor, tif):
    # Reproject to Web Mercator so the image lines up with the basemap, reading a downsampled version
    with rasterio.open(tif) as src, WarpedVRT(src, crs="EPSG:3857", nodata=np.nan) as vrt:
        scale = max(1, max(vrt.width, vrt.height) / MAX_PX)
        h, w = int(vrt.height / scale), int(vrt.width / scale)
        data = vrt.read(1, out_shape=(h, w), resampling=Resampling.average)
        west, south, east, north = transform_bounds(vrt.crs, "EPSG:4326", *vrt.bounds)

    # Display in dB (convert if the tif was written as linear intensity)
    if "_dB" not in tif.stem:
        with np.errstate(divide="ignore", invalid="ignore"):
            data = np.where(data > 0, 10 * np.log10(data), np.nan)

    # 2-98% stretch to grayscale; nodata/outside-scene pixels transparent
    valid = np.isfinite(data)
    vmin, vmax = np.percentile(data[valid], [2, 98])
    stretched = np.clip((np.where(valid, data, vmin) - vmin) / (vmax - vmin), 0, 1)
    gray = (stretched * 255).astype("uint8")
    alpha = (valid * 255).astype("uint8")

    # Encode as an in-memory PNG (gray + alpha) for the overlay
    with warnings.catch_warnings(), MemoryFile() as mem:
        warnings.simplefilter("ignore", NotGeoreferencedWarning)
        with mem.open(driver="PNG", width=w, height=h, count=2, dtype="uint8") as png:
            png.write(np.stack([gray, alpha]))
        url = "data:image/png;base64," + base64.b64encode(mem.read()).decode()

    label = scene_label(sensor, tif)
    print(f"  {label}  {tif.name}: {vmin:.1f} to {vmax:.1f} dB")
    return ImageOverlay(
        url=url,
        bounds=((south, west), (north, east)),
        name=label,
        attribution=ALOS2_ATTRIBUTION if sensor == "ALOS-2" else "",
    )


overlays = [tif_to_overlay(sensor, t) for sensor, t in scenes]

# Center on the AOI point if there is one, otherwise on the newest scene
(south, west), (north, east) = overlays[-1].bounds
center = ((south + north) / 2, (west + east) / 2)
point = SEARCH_AOIS[AOI].get("point")
if point:
    # AOI point is stored as "lon,lat"; Leaflet wants (lat, lon)
    lon, lat = (float(v) for v in point.split(","))
    center = (lat, lon)

m = Map(
    center=center,
    zoom=10,
    basemap=basemaps.Esri.WorldImagery,
    scroll_wheel_zoom=True,
    layout=Layout(height="800px"),
)
# All scenes on, oldest first so the newest is drawn on top; uncheck in the layers control to see the ones underneath
for overlay in overlays:
    m.add(overlay)
if point:
    m.add(Marker(location=center, draggable=False, title=AOI, name=AOI))

m.add(LayersControl(collapsed=False))
m.add(FullScreenControl())
m